In [1]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/citylighxts/dataset-final/dataset_final.csv


### 1. Preprocessing Text + Custom Tokenizer Emoji

In [2]:
!pip install torch torchvision torchaudio sentencepiece -q
print("Dependencies ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

In [3]:
import os
import re
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("All imports OK.")


All imports OK.


In [4]:
MODEL_NAME = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

raw_emoji_to_tag = {
    "😛": "<Wajah Menjulurkan Lidah>",
    "😠": "<Wajah Marah>",
    "💣": "<Bom>",
    "💔": "<Hati Patah>",
    "😕": "<Wajah Bingung>",
    "😞": "<Wajah Kecewa>",
    "😑": "<Wajah Tanpa Ekspresi>",
    "😋": "<Wajah Menikmati Makanan Lezat>",
    "😱": "<Wajah Berteriak Ketakutan>",
    "😓": "<Wajah Dengan Keringat Dingin>",
    "😮": "<Wajah Terperangah Terbuka>",
    "😤": "<Wajah Mengeluarkan Uap Dari Hidung>",
    "😝": "<Wajah Menjulurkan Lidah Dan Mata Tertutup>",
    "😶": "<Wajah Tanpa Mulut>",
    "🔥": "<Api>",
    "☹": "<Wajah Merengut>",
    "😬": "<Wajah Meringis>",
    "⚡": "<Tegangan Tinggi>",
    "🤥": "<Wajah Berbohong>",
    "😣": "<Wajah Bertahan Menderita>",
    "🙇": "<Orang Membungkuk>",
    "🏃": "<Orang Berlari>",
    "🐽": "<Hidung Babi>",
    "😡": "<Wajah Bersungut Marah>",
    "🙈": "<Monyet Menutup Mata>",
    "🙁": "<Wajah Agak Merengut>",
    "🙊": "<Monyet Menutup Mulut>",
    "🤔": "<Wajah Berpikir>",
    "👎": "<Jempol Ke Bawah>",
    "👅": "<Lidah>",
    "😩": "<Wajah Lelah Lesu>",
    "🤐": "<Wajah Mulut Resleting>",
    "😐": "<Wajah Netral>",
    "🙄": "<Wajah Memutar Mata>",
    "😏": "<Wajah Tersenyum Sinis>",
    "😥": "<Wajah Kecewa Tapi Lega>",
    "😯": "<Wajah Terbungkam>",
    "😪": "<Wajah Mengantuk>",
    "😫": "<Wajah Sangat Lelah>",
    "😴": "<Wajah Tidur>",
    "😌": "<Wajah Lega>",
    "😜": "<Wajah Menjulurkan Lidah Dan Mengedipkan Mata>",
    "🤤": "<Wajah Mengiler>",
    "😒": "<Wajah Tidak Terhibur>",
    "😔": "<Wajah Termenung Sedih>",
    "🙃": "<Wajah Terbalik>",
    "🤑": "<Wajah Mata Duitan>",
    "😲": "<Wajah Terperanjat>",
    "😖": "<Wajah Jengkel Bingung>",
    "😟": "<Wajah Cemas>",
    "😢": "<Wajah Menangis>",
    "😭": "<Wajah Menangis Kencang>",
    "😦": "<Wajah Merengut Dengan Mulut Terbuka>",
    "😧": "<Wajah Menderita>",
    "😨": "<Wajah Takut>",
    "😰": "<Wajah Mulut Terbuka Dan Keringat Dingin>",
    "😳": "<Wajah Memerah Tersipu>",
    "😵": "<Wajah Pusing>",
    "😷": "<Wajah Masker Medis>",
    "🤒": "<Wajah Dengan Termometer>",
    "🤕": "<Wajah Dengan Perban Di Kepala>",
    "🤢": "<Wajah Mual>",
    "🤧": "<Wajah Bersin>",
    "🤓": "<Wajah Kutu Buku>",
    "😈": "<Wajah Tersenyum Berandalan>",
    "👿": "<Wajah Marah Bertanduk>",
    "👹": "<Raksasa>",
    "👺": "<Jin>",
    "💀": "<Tengkorak>",
    "☠": "<Tengkorak Dan Tulang Silang>",
    "👻": "<Hantu>",
    "💩": "<Kotoran>",
    "🙀": "<Wajah Kucing Letih>",
    "😿": "<Wajah Kucing Menangis>",
    "😾": "<Wajah Kucing Bersungut>",
    "🙉": "<Monyet Menutup Telinga>",
    "🙎": "<Orang Bersungut>",
    "🙅": "<Orang Isyarat Tidak>",
    "💁": "<Orang Menengadahkan Tangan>",
    "🤦": "<Orang Tepok Jidat>",
    "🤷": "<Orang Mengangkat Bahu>",
    "🤞": "<Jari Menyilang>",
    "📉": "<Grafik Menurun>",
    "⛔": "<Dilarang Masuk>",
    "✖": "<Tanda Silang Perkalian>",
    "❌": "<Tanda Silang>",
    "❎": "<Tombol Tanda Silang>",
    "👌": "<Tangan Isyarat Ok>",
    "👊": "<Tinjuan Ke Depan>",
    "🤘": "<Isyarat Metal>",
    "😍": "<Wajah Tersenyum Dengan Mata Hati>",
    "😊": "<Wajah Tersenyum Dengan Mata Tersenyum>",
    "👍": "<Jempol Ke Atas>",
    "😹": "<Wajah Kucing Menangis Bahagia>",
    "👏": "<Tepuk Tangan>",
    "😘": "<Wajah Meniup Ciuman>",
    "😂": "<Wajah Menangis Gembira>",
    "🙏": "<Tangan Merapat Berdoa>",
    "✊": "<Kepalan Tangan Kejayaan>",
    "🌟": "<Bintang Bersinar>",
    "😁": "<Wajah Menyeringai Dengan Mata Tersenyum>",
    "😀": "<Wajah Menyeringai>",
    "💘": "<Hati Dengan Panah>",
    "✔": "<Tanda Centang Tebal>",
    "🤗": "<Wajah Memeluk>",
    "😚": "<Wajah Mencium Dengan Mata Tertutup>",
    "❤": "<Hati Merah>",
    "🙋": "<Orang Mengangkat Tangan>",
    "🙌": "<Mengangkat Kedua Tangan>",
    "🤣": "<Tertawa Terpingkal-Pingkal>",
    "😆": "<Wajah Tersenyum Mulut Terbuka Dan Mata Tertutup>",
    "😅": "<Wajah Tersenyum Mulut Terbuka Dan Keringat Dingin>",
    "😄": "<Wajah Tersenyum Mulut Terbuka Dan Mata Tersenyum>",
    "😎": "<Wajah Tersenyum Dengan Kacamata Hitam>",
    "🏆": "<Piala>",
    "✌": "<Tangan Tanda Damai Berjaya>",
    "😃": "<Wajah Tersenyum Dengan Mulut Terbuka>",
    "😉": "<Wajah Mengedipkan Mata>",
    "😗": "<Wajah Mencium>",
    "😙": "<Wajah Mencium Dengan Mata Tersenyum>",
    "☺": "<Wajah Tersenyum Simpel>",
    "🙂": "<Wajah Agak Tersenyum>",
    "😇": "<Wajah Tersenyum Malaikat>",
    "🤠": "<Wajah Topi Koboi>",
    "🤡": "<Wajah Badut>",
    "😺": "<Wajah Kucing Tersenyum Mulut Terbuka>",
    "😸": "<Wajah Kucing Menyeringai Mata Tersenyum>",
    "😻": "<Wajah Kucing Tersenyum Mata Hati>",
    "😼": "<Wajah Kucing Tersenyum Kecut>",
    "😽": "<Wajah Kucing Mencium Mata Tertutup>",
    "👨‍🎓": "<Mahasiswa Laki-Laki>",
    "👩‍🎓": "<Mahasiswi Perempuan>",
    "✈": "<Pesawat Terbang>",
    "👼": "<Bayi Malaikat>",
    "👯🏻": "<Wanita Dengan Telinga Kelinci>",
    "🙆🏻": "<Orang Isyarat Ok>",
    "💆": "<Orang Dipijat>",
    "🚶": "<Orang Berjalan>",
    "💃": "<Wanita Menari>",
    "👭": "<Dua Wanita Berpegangan Tangan>",
    "💏": "<Ciuman Romantis>",
    "💑": "<Pasangan Kasih Sayang>",
    "💪": "<Otot Lengan Kuat>",
    "🖐": "<Tangan Terbuka Jari Merenggang>",
    "🤝": "<Berjabat Tangan>",
    "💋": "<Tanda Kecupan Ciuman>",
    "💞": "<Hati Berputar>",
    "💝": "<Hati Dengan Pita Hadiah>",
    "💎": "<Batu Permata Berlian>",
    "🐥": "<Anak Ayam Menghadap Depan>",
    "💐": "<Buket Bunga>",
    "🌹": "<Bunga Mawar>",
    "🌛": "<Bulan Sabit Awal Berwajah>",
    "🌜": "<Bulan Sabit Akhir Berwajah>",
    "🌝": "<Bulan Purnama Berwajah>",
    "🌞": "<Matahari Berwajah>",
    "⭐": "<Bintang Putih Sedang>",
    "🌈": "<Pelangi>",
    "🎀": "<Pita>",
    "🎁": "<Kado Hadiah>",
    "💡": "<Bohlam Lampu Ide>",
    "📈": "<Grafik Meningkat>",
    "💯": "<Nilai Seratus Sempurna>",
    "🆗": "<Tombol Ok>",
    "👨🏽‍❤️‍💋‍👩🏼": "<Ciuman Pria Dan Wanita>",
}

emoji_to_tag_dict = {emoji: tag.lower() for emoji, tag in raw_emoji_to_tag.items()}

custom_tokens = list(emoji_to_tag_dict.values())
tokenizer.add_tokens(custom_tokens)

proposed_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    classifier_dropout=0.3,
    use_safetensors=True,
)
proposed_model.resize_token_embeddings(len(tokenizer))

print(f"Berhasil mendaftarkan {len(custom_tokens)} token emoji custom ke XLM-RoBERTa tokenizer!")
print(f"Model: XLMRobertaForSequenceClassification | num_labels=3 | classifier_dropout=0.3")


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean 

Berhasil mendaftarkan 165 token emoji custom ke XLM-RoBERTa tokenizer!
Model: XLMRobertaForSequenceClassification | num_labels=3 | classifier_dropout=0.3


In [5]:
df_proses = pd.read_csv("/kaggle/input/datasets/citylighxts/dataset-final/dataset_final.csv")

# Hapus kolom yang tidak diperlukan kalau ada
for col in ['tweet emoji', 'Unnamed: 0']:
    if col in df_proses.columns:
        df_proses = df_proses.drop(columns=[col])

# ── Stopword: aman dihapus (bukan negasi/intensifier/partikel sentimen) ──
# JANGAN hapus: tidak, bukan, belum, jangan, tak, ga, gak, nggak (negasi)
# JANGAN hapus: sangat, banget, sekali, amat, terlalu, parah (intensifier)
# JANGAN hapus: sih, deh, dong, nih, lho, kan (partikel tone/sentimen)
STOPWORDS = {
    'yang', 'di', 'ke', 'dari', 'dengan', 'untuk', 'ini', 'itu',
    'oleh', 'pada', 'atau', 'dan', 'juga', 'telah', 'ada', 'adalah',
    'pun', 'pula', 'saja', 'lah', 'kah', 'si', 'sang', 'para',
    'maka', 'agar', 'supaya', 'karena', 'sebab', 'jika', 'kalau',
    'bila', 'apabila', 'walaupun', 'meskipun', 'setelah', 'sesudah',
    'sebelum', 'ketika', 'saat', 'waktu', 'seperti', 'antara',
    'dalam', 'semua', 'setiap', 'tiap', 'masing', 'bisa', 'dapat',
    'boleh', 'perlu', 'sedang', 'kami', 'kita', 'mereka', 'dia',
    'ia', 'anda', 'saya', 'aku', 'kamu', 'kalian', 'nya', 'mu', 'ku',
}

def clean_preprocessing(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'https?://\s*\S+|www\.\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'[^a-z\s<>]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Stopword removal: pisahkan tag emoji <...> agar tidak terganggu
    parts = re.split(r'(<[^>]*>)', text)
    filtered = []
    for part in parts:
        if re.match(r'<[^>]*>', part):
            filtered.append(part)
        else:
            tokens = [t for t in part.split() if t not in STOPWORDS]
            filtered.append(' '.join(tokens))
    text = re.sub(r'\s+', ' ', ' '.join(filtered)).strip()
    return text

df_proses['tweet_clean'] = df_proses['tweet_with_tags'].apply(clean_preprocessing)

# Filter teks yang terlalu pendek setelah cleaning (< 2 token)
sebelum = len(df_proses)
df_proses = df_proses[df_proses['tweet_clean'].str.split().str.len() >= 2].reset_index(drop=True)
print(f"Filter teks pendek: {sebelum} → {len(df_proses)} baris (hapus {sebelum - len(df_proses)})")
print(f"\nTotal data: {len(df_proses)} baris")
print(df_proses[['tweet_with_tags', 'tweet_clean', 'sentimen llm']].head())


Filter teks pendek: 23674 → 23672 baris (hapus 2)

Total data: 23672 baris
                                     tweet_with_tags  \
0  wah belom liat muka gue lagi murka hahahaha <W...   
1  Mungkin kurang piknik adrenalin <Wajah Memutar...   
2  berserah pada maha esa paling tabah dan sabar ...   
3  Ehekk malu la hahahahahahaha <Monyet Menutup M...   
4  WKWKWKWK KESAL AKU BACANYA TAPI KOK SENYUM2 <W...   

                                         tweet_clean sentimen llm  
0  wah belom liat muka gue lagi murka hahahaha <w...      negatif  
1  mungkin kurang piknik adrenalin <wajah memutar...      negatif  
2  berserah maha esa paling tabah sabar selawat z...      positif  
3  ehekk malu la hahahahahahaha <monyet menutup m...      positif  
4  wkwkwkwk kesal bacanya tapi kok senyum <wajah ...      positif  


### 2. Inspeksi Data

In [6]:
print("Jumlah data awal per kelas sentimen:")
print(df_proses['sentimen llm'].value_counts())

print(f"\nTotal data sebelum penyeimbangan kelas: {len(df_proses)} baris")

Jumlah data awal per kelas sentimen:
sentimen llm
negatif    10477
netral      7763
positif     5432
Name: count, dtype: int64

Total data sebelum penyeimbangan kelas: 23672 baris


In [7]:
label_mapping = {'negatif': 0, 'netral': 1, 'positif': 2}
df_proses['label'] = df_proses['sentimen llm'].map(label_mapping)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=df_proses['label'].values
)
class_weights_tensor = torch.FloatTensor(class_weights)

print("Distribusi kelas asli (semua data dipakai, tidak undersample):")
print(df_proses['sentimen llm'].value_counts())
print(f"\nTotal data: {len(df_proses)} baris")
print(f"\nClass weights → negatif: {class_weights[0]:.4f} | netral: {class_weights[1]:.4f} | positif: {class_weights[2]:.4f}")


Distribusi kelas asli (semua data dipakai, tidak undersample):
sentimen llm
negatif    10477
netral      7763
positif     5432
Name: count, dtype: int64

Total data: 23672 baris

Class weights → negatif: 0.7531 | netral: 1.0164 | positif: 1.4526


In [8]:
from sklearn.model_selection import train_test_split

# Gunakan semua data (df_proses), bukan df_balanced
train_df, val_df = train_test_split(
    df_proses,
    test_size=0.2,
    random_state=42,
    stratify=df_proses['label']
)

print(f"Data Train: {len(train_df)} baris")
print(f"Data Val  : {len(val_df)} baris")
print("\nDistribusi kelas di Data Train:")
print(train_df['sentimen llm'].value_counts())

Data Train: 18937 baris
Data Val  : 4735 baris

Distribusi kelas di Data Train:
sentimen llm
negatif    8381
netral     6210
positif    4346
Name: count, dtype: int64


### 3. Setup Model

In [9]:
class SentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row['tweet_clean']
        label = row['label']
        
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }


In [10]:
BATCH_SIZE = 4  # Diturunkan 8 → 4 karena max_len 256 butuh lebih banyak VRAM

train_loader = DataLoader(SentimentDataset(train_df, tokenizer), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SentimentDataset(val_df, tokenizer), batch_size=BATCH_SIZE, shuffle=False)

print("DataLoader berhasil diinisialisasi!")
sample_batch = next(iter(train_loader))
print(f"Ukuran tensor Input IDs per batch: {sample_batch['input_ids'].shape}")
print(f"Ukuran tensor Labels per batch: {sample_batch['label'].shape}")


DataLoader berhasil diinisialisasi!
Ukuran tensor Input IDs per batch: torch.Size([4, 256])
Ukuran tensor Labels per batch: torch.Size([4])


### 4. Setup Parameter

In [11]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Device yang digunakan: {device}")

proposed_model = proposed_model.to(device)

# Full fine-tuning: unfreeze SEMUA layer
for param in proposed_model.parameters():
    param.requires_grad = True

# Differential LR: backbone kecil agar tidak merusak pretrained weights
bert_params = list(proposed_model.roberta.parameters())
head_params  = list(proposed_model.classifier.parameters())

optimizer = torch.optim.AdamW([
    {'params': bert_params, 'lr': 2e-6},
    {'params': head_params,  'lr': 1e-5},
], weight_decay=0.02)

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor.to(device),
    label_smoothing=0.05
)

trainable = sum(p.numel() for p in proposed_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in proposed_model.parameters())
print(f"Full fine-tuning: semua {trainable:,} / {total:,} params di-unfreeze ({100*trainable/total:.1f}%)")
print("Differential LR → BERT backbone: 2e-6 | Classifier head: 1e-5")
print("weight_decay: 0.02 | label_smoothing: 0.05 | classifier_dropout: 0.3 | max_len: 256")


Device yang digunakan: cuda
Full fine-tuning: semua 278,172,675 / 278,172,675 params di-unfreeze (100.0%)
Differential LR → BERT backbone: 2e-6 | Classifier head: 1e-5
weight_decay: 0.02 | label_smoothing: 0.05 | classifier_dropout: 0.3 | max_len: 256


### 5. Training

In [12]:
NUM_EPOCHS = 40
PATIENCE = 5
CHECKPOINT_PATH = "best_model.pt"

scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

print("Memulai Training & Validasi Baseline 3 (Proposed Method)...")
print("-" * 75)

history_records = []
best_val_f1 = 0.0
epochs_no_improve = 0

for epoch in range(NUM_EPOCHS):
    start_time = time.time()

    # ── TRAINING ──
    proposed_model.train()
    total_train_loss = 0
    all_train_preds, all_train_labels = [], []

    for batch in train_loader:
        optimizer.zero_grad()
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        outputs = proposed_model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits

        loss = criterion(logits, labels)
        total_train_loss += loss.item()

        _, preds = torch.max(logits, dim=1)
        all_train_preds.extend(preds.cpu().numpy())
        all_train_labels.extend(labels.cpu().numpy())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(proposed_model.parameters(), max_norm=1.0)
        optimizer.step()

    scheduler.step()
    current_lr = scheduler.get_last_lr()

    avg_train_loss = total_train_loss / len(train_loader)
    train_acc  = accuracy_score(all_train_labels, all_train_preds)
    train_prec, train_rec, train_f1, _ = precision_recall_fscore_support(
        all_train_labels, all_train_preds, average='macro', zero_division=0
    )

    # ── VALIDASI ──
    proposed_model.eval()
    total_val_loss = 0
    all_val_preds, all_val_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = proposed_model(input_ids=input_ids, attention_mask=attention_mask)
            logits  = outputs.logits

            loss = criterion(logits, labels)
            total_val_loss += loss.item()

            _, preds = torch.max(logits, dim=1)
            all_val_preds.extend(preds.cpu().numpy())
            all_val_labels.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)
    val_acc  = accuracy_score(all_val_labels, all_val_preds)
    val_prec, val_rec, val_f1, _ = precision_recall_fscore_support(
        all_val_labels, all_val_preds, average='macro', zero_division=0
    )

    elapsed = time.time() - start_time

    # ── Checkpoint & Early Stopping ──
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        epochs_no_improve = 0
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': proposed_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': val_f1,
            'val_acc': val_acc,
        }, CHECKPOINT_PATH)
        marker = " ← best (saved)"
    else:
        epochs_no_improve += 1
        marker = f" (no improve {epochs_no_improve}/{PATIENCE})"

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | {elapsed:.2f}s | LR: {current_lr[0]:.2e}")
    print(f"  [TRAIN] Loss: {avg_train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"  [VALID] Loss: {avg_val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}{marker}")
    print("-" * 75)

    history_records.append({
        'epoch': epoch + 1,
        'train_loss': avg_train_loss, 'train_accuracy': train_acc,
        'train_precision': train_prec, 'train_recall': train_rec, 'train_f1_score': train_f1,
        'val_loss': avg_val_loss, 'val_accuracy': val_acc,
        'val_precision': val_prec, 'val_recall': val_rec, 'val_f1_score': val_f1
    })

    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping triggered! Val F1 tidak improve selama {PATIENCE} epoch berturut-turut.")
        break

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
proposed_model.load_state_dict(checkpoint['model_state_dict'])
print(f"\nTraining selesai! Best Val F1: {best_val_f1:.4f} (epoch {checkpoint['epoch']})")
print(f"Best model di-load dari: '{CHECKPOINT_PATH}'")

df_history = pd.DataFrame(history_records)
df_history.to_csv("history_validation.csv", index=False)
print(f"History disimpan di: 'history_validation.csv'")


Memulai Training & Validasi Baseline 3 (Proposed Method)...
---------------------------------------------------------------------------
Epoch [1/40] | 1337.20s | LR: 2.00e-06
  [TRAIN] Loss: 0.9702 | Acc: 0.5730 | F1: 0.5521
  [VALID] Loss: 0.8036 | Acc: 0.6870 | F1: 0.6753 ← best (saved)
---------------------------------------------------------------------------
Epoch [2/40] | 1339.31s | LR: 1.99e-06
  [TRAIN] Loss: 0.8221 | Acc: 0.6953 | F1: 0.6814
  [VALID] Loss: 0.7829 | Acc: 0.7238 | F1: 0.7116 ← best (saved)
---------------------------------------------------------------------------
Epoch [3/40] | 1339.59s | LR: 1.99e-06
  [TRAIN] Loss: 0.7637 | Acc: 0.7321 | F1: 0.7189
  [VALID] Loss: 0.7542 | Acc: 0.7518 | F1: 0.7382 ← best (saved)
---------------------------------------------------------------------------
Epoch [4/40] | 1339.36s | LR: 1.98e-06
  [TRAIN] Loss: 0.7237 | Acc: 0.7624 | F1: 0.7503
  [VALID] Loss: 0.7747 | Acc: 0.7645 | F1: 0.7519 ← best (saved)
--------------------